## Transferring data between tables

This notebook allows the user to move the entries of specified columns from a source table into a target table. This is useful if a pipeline step had to be re-done due to a bug or methodological change, but you don't want to re-run the entire pipeline.

In [ ]:
from astropy.io import fits
import astropy.table as aptb
import numpy as np

from pathlib import Path

In [ ]:
SPEC_TYPE = "2fwhm_opt"

# specify the source and target file paths
tables_dir = Path("./megatables")
source_name = f"lae_megatab_flagged_cpts_refit_{SPEC_TYPE}.fits"
source_path = tables_dir / source_name
target_name = f"lae_megatab_flagged_cpts_allrefit_zeldamcmc_sysz_{SPEC_TYPE}.fits"
target_path = tables_dir / target_name

In [ ]:
# Specify columns to be updated in the target table
columns_to_update = [
    "AMPB",
    "AMPB_ERR",
    "FWHMB",
    "FWHMB_ERR",
    "DISPB",
    "DISPB_ERR",
    "FLUXB",
    "FLUXB_ERR",
    "ASYMB",
    "ASYMB_ERR",
    "LPEAKB",
    "LPEAKB_ERR",
    "SNRB",
]

In [ ]:
# Open the tables
with fits.open(source_path) as source_hdul, fits.open(target_path) as target_hdul:
    source_table = aptb.Table(source_hdul[1].data)
    target_table = aptb.Table(target_hdul[1].data)

    # Create a mapping from source table rows to target table rows based on both 'iden' and 'CLUSTER' columns
    source_iden_cluster = list(zip(source_table["iden"], source_table["CLUSTER"]))
    target_iden_cluster = list(zip(target_table["iden"], target_table["CLUSTER"]))
    mapping = {source_iden_cluster[i]: i for i in range(len(source_iden_cluster))}

    # Update the target table with values from the source table for the specified columns
    for i, target_row in enumerate(target_table):
        key = (target_row["iden"], target_row["CLUSTER"])
        if key in mapping:
            source_row = source_table[mapping[key]]
            for col in columns_to_update:
                target_table[col][i] = source_row[col]

    # Save the updated target table back to a new FITS file
    target_table.write(target_path, overwrite=True)